# 초기 분석 변수 정의 단계(단일 배터리사용)

In [2]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import scipy.io

# 데이터 로드
mat = scipy.io.loadmat("../data/NASA/5. Battery Data Set/1. BatteryAgingARC-FY08Q4/B0005.mat")  # 네 파일명 맞게 수정
cycle = mat["B0005"][0,0]["cycle"][0]

print(len(cycle))

616


In [3]:
# 1. discharge
capacities = []
for record in cycle:
    if record["type"][0] == "discharge":
        cap = record["data"][0,0]["Capacity"][0,0]
        capacities.append(cap)

# 2. impedance
Rct_list = []
Re_list = []
for record in cycle:
    if record["type"][0] == "impedance":
        data = record["data"][0,0]
        Rct_list.append(data["Rct"][0,0])
        Re_list.append(data["Re"][0,0])

In [4]:
min_len = min(len(capacities), len(Rct_list))

capacities = capacities[:min_len]
Rct_list   = Rct_list[:min_len]
Re_list    = Re_list[:min_len]

In [5]:
df = pd.DataFrame({
    "Cycle": range(1, min_len+1),
    "Capacity": capacities,
    "Rct": Rct_list,
    "Re": Re_list
})

In [6]:
df.head()

,Cycle,Capacity,Rct,Re
0,1,1.856487,0.069456,0.044669
1,2,1.846327,0.076275,0.046687
2,3,1.835349,0.067972,0.044843
3,4,1.835263,0.074534,0.046195
4,5,1.834646,0.068528,0.045101


In [7]:
df["SOH"] =  df["Capacity"] / df["Capacity"].iloc[0]

In [8]:
#시상수
df["tau_real"] = df["Rct"] * df["Capacity"]

In [9]:
#CCA_like 후보, CCA: Capacity비례, Rct 반비례
df["CCA1"] = df["Capacity"] / df["Rct"]
df["CCA2"] = df["Capacity"]
df["CCA3"] = 1 / df["Rct"]
df["CCA4"] = df["Capacity"] / (df["Rct"]**0.5)

In [10]:
# τ_proxy
df["tau_p1"] = df["Rct"] * df["CCA1"]
df["tau_p2"] = df["Rct"] * df["CCA2"]
df["tau_p3"] = df["Rct"] * df["CCA3"]
df["tau_p4"] = df["Rct"] * df["CCA4"]

In [11]:
df[[
    "tau_real",
    "tau_p1",
    "tau_p2",
    "tau_p3",
    "tau_p4"
]].corr()

,tau_real,tau_p1,tau_p2,tau_p3,tau_p4
tau_real,1.000000,0.868439,1.000000,0.296014,0.950860
tau_p1,0.868439,1.000000,0.868439,0.320882,0.979252
tau_p2,1.000000,0.868439,1.000000,0.296014,0.950860
tau_p3,0.296014,0.320882,0.296014,1.000000,0.318085
tau_p4,0.950860,0.979252,0.950860,0.318085,1.000000


In [22]:
df.to_csv("nasa_feature.csv", index=False)